In [1]:
import os
print(os.getcwd())

/Users/utopiaideal/Desktop/desktop/coding/codeInvest/test


In [ ]:
# 1. 获取当前的工作目录
current_wd = os.getcwd()
print(f"当前目录: {current_wd}")
# 2. 获取上一级目录
# os.path.dirname(路径) 会自动剥离掉最后一层目录名，即回到上一级
parent_dir = os.path.dirname(current_wd)
# 或者用这种写法，更直观（".." 代表上一级）
# parent_dir = os.path.abspath(os.path.join(current_wd, ".."))
# 3. 切换工作目录
os.chdir(parent_dir)
print(f"已切换到上级目录: {os.getcwd()}")

当前目录: /Users/utopiaideal/Desktop/desktop/coding/codeInvest/test
已切换到上级目录: /Users/utopiaideal/Desktop/desktop/coding/codeInvest


In [8]:
import logging

# 1. 导入接口定义的异常和抽象基类
from src.api import (
    FinancialDataSource, 
    NoDataFoundError, 
    DataSourceError
)

# 2. 导入具体的实现类
from src.api import BaostockDataSource

# 配置日志，以便观察 _execute_query 内部的运行情况
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - [%(levelname)s] - %(message)s'
)
logger = logging.getLogger(__name__)

def run_test():
    print("="*60)
    print("开始接口实现测试: FinancialDataSource -> BaostockDataSource")
    print("="*60)

    # ---------------------------------------------------------
    # 核心步骤: 实例化
    # 虽然 Python 是动态语言，但我们可以通过类型注解强调这是接口的实现
    # ---------------------------------------------------------
    data_source: FinancialDataSource = BaostockDataSource()
    
    print(f"\n[检查点] 实例类型: {type(data_source)}")
    print(f"[检查点] 是否是 FinancialDataSource 的实例? {isinstance(data_source, FinancialDataSource)}")

    # ---------------------------------------------------------
    # 测试场景 1: 获取历史 K 线数据 (基础功能)
    # ---------------------------------------------------------
    print("\n>>> 测试 1: 获取日线数据 (get_historical_k_data)")
    try:
        # 查询 浦发银行 (sh.600000) 2023年1月的行情
        df_k = data_source.get_historical_k_data(
            code="sh.600000",
            start_date="2023-01-01",
            end_date="2023-01-10",
            frequency="d",
            adjust_flag="3"
        )
        print(f"✅ 成功获取数据，共 {len(df_k)} 行。前3行如下:")
        print(df_k[['date', 'code', 'close', 'pctChg']].head(3))
        
    except Exception as e:
        print(f"❌ 测试 1 失败: {e}")

    # ---------------------------------------------------------
    # 测试场景 2: 获取财务报表 (验证 _execute_query 的通用性)
    # ---------------------------------------------------------
    print("\n>>> 测试 2: 获取季频盈利能力 (get_profit_data)")
    try:
        # 查询 贵州茅台 (sh.600519) 2022年第一季度
        df_profit = data_source.get_profit_data(
            code="sh.600519",
            year="2022",
            quarter=1
        )
        # 检查是否包含关键字段 'roeAvg'
        roe = df_profit['roeAvg'].values[0] if not df_profit.empty else "N/A"
        print(f"✅ 成功获取茅台盈利数据。平均ROE: {roe}")
        
    except Exception as e:
        print(f"❌ 测试 2 失败: {e}")

    # ---------------------------------------------------------
    # 测试场景 3: 异常处理 (验证 NoDataFoundError)
    # ---------------------------------------------------------
    print("\n>>> 测试 3: 查询不存在的股票 (错误处理测试)")
    try:
        # 一个不存在的代码
        data_source.get_stock_basic_info(code="sh.999999")
        print("❌ 失败: 应该报错但没有报错！")
    except NoDataFoundError as e:
        print(f"✅ 成功捕获预期异常 (NoDataFoundError): {e}")
    except Exception as e:
        print(f"❌ 失败: 捕获了错误的异常类型: {type(e)}")

    # ---------------------------------------------------------
    # 测试场景 4: 宏观数据 (验证不同参数签名的调用)
    # ---------------------------------------------------------
    print("\n>>> 测试 4: 获取存款利率 (get_deposit_rate_data)")
    try:
        df_rate = data_source.get_deposit_rate_data(
            start_date="2023-01-01", 
            end_date="2023-12-31"
        )
        print(f"✅ 成功获取宏观数据，共 {len(df_rate)} 条记录。")
    except Exception as e:
        print(f"❌ 测试 4 失败: {e}")

    print("\n" + "="*60)
    print("测试结束")
    print("="*60)


In [9]:
run_test()

2026-01-19 21:26:28,633 - [INFO] - Fetching Historical K-Data with params: {'code': 'sh.600000', 'fields': 'date,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,peTTM,pbMRQ,psTTM,pcfNcfTTM,isST', 'start_date': '2023-01-01', 'end_date': '2023-01-10', 'frequency': 'd', 'adjustflag': '3'}


开始接口实现测试: FinancialDataSource -> BaostockDataSource

[检查点] 实例类型: <class 'src.api.baostock_data_source.BaostockDataSource'>
[检查点] 是否是 FinancialDataSource 的实例? True

>>> 测试 1: 获取日线数据 (get_historical_k_data)
login success!


2026-01-19 21:26:30,927 - [INFO] - Retrieved 6 records for Historical K-Data.


logout success!


2026-01-19 21:26:31,050 - [INFO] - Fetching Profitability with params: {'code': 'sh.600519', 'year': '2022', 'quarter': 1}


...

logout success!


2026-01-19 21:26:32,304 - [INFO] - Fetching Deposit Rate with params: {'start_date': '2023-01-01', 'end_date': '2023-12-31'}


✅ 成功捕获预期异常 (NoDataFoundError): No Stock Basic Info found (empty result).

>>> 测试 4: 获取存款利率 (get_deposit_rate_data)
login success!


2026-01-19 21:26:32,859 - [WARNING] - Empty result set for Deposit Rate with params: {'start_date': '2023-01-01', 'end_date': '2023-12-31'}


logout success!
❌ 测试 4 失败: No Deposit Rate found (empty result).

测试结束
